In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from itertools import count
import pandas as pd
import os
from tqdm.notebook import tqdm
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from pypdf import PdfReader
import chromadb
from llama_index.core.node_parser.text import SentenceSplitter
from dotenv import load_dotenv
import os
import xml.etree.ElementTree as ET
from typing import Iterator, Tuple
import chromadb
from more_itertools import chunked
from snomed_graph.snomed_graph import SnomedGraph, SnomedConceptDetails

In [ ]:
load_dotenv()
CHUNK_SIZE_CHARS = 1000

# Download and process monolingual web resources

In [ ]:
def download_pdf_if_pdf(url, save_dir):
    """
    Downloads the PDF from the given URL and saves it to the specified directory
    using the last segment of the URL as the filename.
    If the URL returns an HTML page or any non-PDF content, it does nothing.
    """
    # Ensure the target directory exists.
    os.makedirs(save_dir, exist_ok=True)
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {url}: {e}")
        return False

    # Check the Content-Type header.
    content_type = response.headers.get("Content-Type", "").lower()
    if "application/pdf" in content_type:
        # Use the last segment of the URL path as the filename.
        parsed_url = urlparse(url)
        filename = os.path.basename(parsed_url.path)
        if not filename:
            print("No valid filename found in URL.")
            return True
        filepath = os.path.join(save_dir, filename + ".pdf")
        
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"Saved PDF to {filepath}")
        return True
    else:
        print(f"Ignored {url}: Content-Type is not PDF.")
    return False

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")  


def extract_text_from_pdf(pdf_path, use_gemini=True):
    """
    Extracts text from a PDF using Gemini 2.0 Flash via LangChain
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Path to save the extracted text
    """
    
    # Load the PDF document
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    
    # Combine all pages into a single text
    pdf_text = "\n".join([page.page_content for page in pages])
        
    if use_gemini:
        # Initialize the Gemini model
        llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
        
        # Create a prompt template
        prompt_template = PromptTemplate(
            input_variables=["pdf_content"],
            template="""
            Extract and return only the main text content from this PDF document.
            Ignore headers, footers, page numbers, appendices, sidebars and other non-content elements.
            Format the text cleanly, preserving paragraphs and important structure.
            
            Document content:
            {pdf_content}
            """
        )

        chain = LLMChain(llm=llm, prompt=prompt_template)
        result = chain.run(pdf_content=pdf_text)
        return result
    else:
        return pdf_text


In [ ]:
def list_files_in_directory(directory_path, suffix="pdf"):
    try:
        # Get all files and folders in the directory
        files = os.listdir(directory_path)
        
        for f in tqdm(files):
            if os.path.isfile(os.path.join(directory_path, f)) and f.endswith(suffix):
                yield os.path.join(directory_path, f)
        
    except Exception as e:
        print(f"Error: {e}")

## Download medical dictionaries from sonaveeb.ee

In [ ]:
# These are the specialist dictionaries on Sonaveeb.ee
dictionaries = [
    'https://sonaveeb.ee/ds/bks',
    'https://sonaveeb.ee/ds/gen',
    'https://sonaveeb.ee/ds/GER',
    'https://sonaveeb.ee/ds/den',
    'https://sonaveeb.ee/ds/imm',
    'https://sonaveeb.ee/ds/lon',
    'https://sonaveeb.ee/ds/mef',
    'https://sonaveeb.ee/ds/nfs',
    'https://sonaveeb.ee/ds/pot',
    'https://sonaveeb.ee/ds/rkb',
    'https://sonaveeb.ee/ds/TAI',
    'https://sonaveeb.ee/ds/glu',
    'https://sonaveeb.ee/ds/%C3%95TERM',
]

In [ ]:
def fetch_sonaveeb_dictionary_subpages(url):
    # Fetch the page
    response = requests.get(url)
    response.raise_for_status()  # will raise an error for bad responses

    # Parse the HTML content
    soup = BeautifulSoup(response.content, "html.parser")

    # Use the response URL as the base for relative links
    base_url = response.url

    # Collect URLs from links whose text is a single letter
    letter_urls = []
    for a in soup.find_all('a'):
        text = a.get_text(strip=True)
        if len(text) == 1 and text.isalpha():
            href = a.get('href')
            # Convert relative URLs to absolute URLs
            full_url = urljoin(base_url, href)
            letter_urls.append(full_url)

    print(f"Dictionary at {url} has {len(letter_urls)} subpages")
    if url == 'https://sonaveeb.ee/ds/mef':
        for idx, l in enumerate(letter_urls):
            if l == 'https://sonaveeb.ee/ds/mef/r':
                letter_urls = letter_urls[idx: ]
    return letter_urls

In [ ]:
def fetch_sonaveeb_dictionary_entries(url):
    # Download the page
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Parse the HTML content
    soup = BeautifulSoup(response.content, "html.parser")

    # Get all <a> tags in document order
    all_links = soup.find_all("a")

    # Find the index of the last letter hyperlink (assumed to be a single alphabetic character)
    last_letter_index = None
    for i, link in enumerate(all_links):
        text = link.get_text(strip=True)
        if len(text) == 1 and text.isalpha():
            last_letter_index = i

    medical_term_urls = []
    if last_letter_index is not None:
        # Iterate over all links that appear after the letter navigation block
        for link in all_links[last_letter_index + 1:]:
            text = link.get_text(strip=True)
            # Filter: we expect medical term links to have more than one character in their text
            if text and len(text) > 1:
                href = link.get("href")
                if href:
                    full_url = urljoin(response.url, href)
                    if full_url.startswith("https://sonaveeb.ee/search"):
                        medical_term_urls.append(full_url)

    print(f"Subpage at {url} has {len(medical_term_urls)} terms")
    return medical_term_urls

In [ ]:
def fetch_sonaveeb_dictionary_definitions(url):
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    try:
        # Extract the word "abiõde" from the nested <span> within the "homonym-name" container.
        title = soup.select_one("span.homonym-name span").get_text(strip=True)

        # Extract the language code "et" from the span with the specific language classes.
        lang = soup.select_one("span.text-muted.mr-1.lang-code.lang-code-sm").get_text(strip=True)

        # Extract the introduction text from the "homonym-intro d-block" span.
        intro = soup.select_one("span.homonym-intro.d-block").get_text(strip=True)

        result = (title, lang, intro)
    except:
        result = None
    return result

In [ ]:
sonaveeb_results = list()

In [ ]:
it = count()

for u1 in tqdm(dictionaries):
    for u2 in tqdm(fetch_sonaveeb_dictionary_subpages(u1)):
        for u3 in fetch_sonaveeb_dictionary_entries(u2):
            result = fetch_sonaveeb_dictionary_definitions(u3)
            if result:
                sonaveeb_results.append(r)
            if next(it) % 10 == 0:
                time.sleep(1)
    print(f"{len(sonaveeb_results)} entries so far")

In [ ]:
pd.DataFrame([
    {'term': term, 'lang': lang, 'desc': desc}
    for term, lang, desc in sonaveeb_results
]).to_csv("../data/sonaveeb.csv", index=False)

## Download medical dictionary from tervisesonastik.tai.ee

In [ ]:
tervisesonastik_results = list()

In [ ]:

def fetch_tervisesonastik_entries(url):
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, 'html.parser')
    entries = []
    
    # The page is structured as a series of "views-row" containers.
    containers = soup.find_all("div", class_="views-row")
    
    for container in containers:
        # 1. Get the term.
        term_span = container.find("span", class_="field field--name-title field--type-string field--label-hidden")
        if term_span is None:
            continue
        term = term_span.get_text(strip=True)
        
        # 2. Get the description.
        # Now, we explicitly look for the ordered list that holds the meanings.
        description = ""
        node_content = container.find("div", class_="node__content")
        if node_content:
            meanings_ol = node_content.find("ol", class_=lambda c: c and "field--name-field-meanings" in c)
            if meanings_ol:
                li = meanings_ol.find("li", class_="field__item")
                if li:
                    # This should join all text (from <em>, <a>, etc.) into a single string.
                    description = li.get_text(" ", strip=True).replace("\xa0", " ")
        
        # 3. Get the source.
        source = None
        source_label = container.find(lambda tag: tag.has_attr("class") and "field__label" in tag.get("class", []) and tag.get_text(strip=True) == "Allikas:")
        if source_label:
            source_div = source_label.find_next("div", class_="field__item")
            if source_div:
                source_link = source_div.find("a")
                if source_link:
                    source = source_link.get("href")
        
        # 4. Get the English alias, removing text within <em> tags.
        english_alias = None
        for label in container.find_all(class_="field__label"):
            if "Teise keele vaste:" in label.get_text():
                alias_item = label.find_next_sibling("div")
                if alias_item:
                    # Remove all <em> tags from the alias block.
                    for em in alias_item.find_all("em"):
                        em.decompose()
                    english_alias = alias_item.get_text(" ", strip=True)
                break
        
        entries.append((term, description, source, english_alias))
    
    return entries


In [ ]:
for grp in tqdm(range(1, 33)):
    url = f"https://tervisesonastik.tai.ee/tervisesonastik?alphabetical_group={grp}"
    entries = fetch_tervisesonastik_entries(url)
    print(f"Group {grp} contained {len(entries)} entries")
    tervisesonastik_results += entries

In [ ]:
pd.DataFrame([
    {'term': term, 'desc': desc, 'source': src, 'english_alias': eng}
    for term, desc, src, eng in tervisesonastik_results
]).to_csv("../data/tervisesonastik.csv", index=False)

## Download the Eesti Arst PDF archive and extract the text

In [ ]:
it1 = count()
it2 = count()

def download_articles(url, suppress=set()):
    suppress.add(url)
    c = next(it1)
    if c % 25 == 0:
        time.sleep(5)        
        
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Find all <a> tags with an href attribute and filter those starting with the prefix.
    for a in soup.find_all('a', href=True):
        href = a['href']
        if href not in suppress:            
            if "/view/" in href:
                yield from download_articles(href)
            if "article/download/" in href:
                if download_pdf_if_pdf(href.replace("/view/", "/download/")):
                    print(f"Visited: {c}.  Downloaded: {next(it2)}")

In [ ]:
archive_url = "https://ojs.utlib.ee/index.php/EA/issue/archive"
list(download_articles(archive_url))

In [ ]:
# Count pages of PDFs
pdf_pages = 0
num_articles = 0

for file in tqdm(os.listdir("../data/eesti_arst")):
    if file[-4:] == ".pdf":
        num_articles +=1 
        try:
            reader = PdfReader("../data/eesti_arst/" + file)
            pdf_pages += len(reader.pages)
        except:
            pass
        
print(f"{pdf_pages} pages from {num_articles} articles.")

In [ ]:
for pdf in list_files_in_directory("../data/eesti_arst"):
    filename = os.path.basename(pdf)
    filename, _ = os.path.splitext(filename)
    if not os.path.exists(f"../data/eesti_arst/{filename}.txt"):
        text = extract_text_from_pdf(pdf)
        with open(f"../data/eesti_arst/{filename}.txt", "w") as f:
            f.write(text)

In [ ]:
# Total cost for all 5,508 PDFs was $7
# What is the cost per PDF page?
# Answer, 200 pages per cent!

7 / pdf_pages

In [ ]:
# How large is the corpus?

words = 0
for file in tqdm(os.listdir("../data/eesti_arst")):
    if file[-4:] == ".txt":
        with open(f"../data/eesti_arst/{file}", "r") as f:
            text = f.read()
            words += len(text.split(" "))
print(words)

## Download and parse patient information leaflets from haiglateliit.ee

In [ ]:
# Input URL of the webpage to scan
target_url = "https://haiglateliit.ee/patsientide-infomaterjalid/"
response = requests.get(target_url)
soup = BeautifulSoup(response.text, "html.parser")
links = soup.find_all("a", href=True)    
print(f"Found {len(links)} links on the page.")

for link in tqdm(links):
    href = link.get("href")
    # Construct an absolute URL from possibly relative href
    full_url = urljoin(target_url, href)
    # Check if URL points to a PDF (case-insensitive check)
    if full_url.lower().endswith(".pdf"):
        try:
            download_pdf_if_pdf(full_url, "../data/haiglateliit")
        except Exception as e:
            print(f"Failed to download {full_url}: {e}")


In [ ]:
for pdf in list_files_in_directory("../data/haiglateliit"):
    filename = os.path.basename(pdf)
    filename, _ = os.path.splitext(filename)
    if not os.path.exists(f"../data/haiglateliit/{filename}.txt"):
        text = extract_text_from_pdf(pdf)
        with open(f"../data/haiglateliit/{filename}.txt", "w") as f:
            f.write(text)

In [ ]:
# How large is the corpus?

words = 0
for file in tqdm(os.listdir("../data/haiglateliit")):
    if file[-4:] == ".txt":
        with open(f"../data/haiglateliit/{file}", "r") as f:
            text = f.read()
            words += len(text.split(" "))
print(words)

## Download and parse laboratory guidelines from kliinikum.ee

In [ ]:
# Input URL of the webpage to scan
base_url = "https://www.kliinikum.ee/yhendlabor/raamat/"
subdirs = [
    "A", "Bc", "De", "Fg", "Hij", "K", "L", "Mn", "Op", "Rs", "T", "Uy", "0Laboriuuringute_algoritmid" 
]
for s in tqdm(subdirs):
    url = base_url + s + "/"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    links = soup.find_all("a", href=True)    
    for link in links:
        href = link.get("href")
        # Construct an absolute URL from possibly relative href
        full_url = urljoin(url, href)
        # Check if URL points to a PDF (case-insensitive check)
        if full_url.lower().endswith(".pdf"):
            try:
                download_pdf_if_pdf(full_url, "../data/kliinikum")
            except Exception as e:
                print(f"Failed to download {full_url}: {e}")


In [ ]:
for pdf in tqdm(list_files_in_directory("../data/kliinikum")):
    filename = os.path.basename(pdf)
    filename, _ = os.path.splitext(filename)
    if not os.path.exists(f"../data/kliinikum/{filename}.txt"):
        text = extract_text_from_pdf(pdf)
        with open(f"../data/kliinikum/{filename}.txt", "w") as f:
            f.write(text)

In [ ]:
# How large is the corpus?

words = 0
for file in tqdm(os.listdir("../data/kliinikum")):
    if file[-4:] == ".txt":
        with open(f"../data/kliinikum/{file}", "r") as f:
            text = f.read()
            words += len(text.split(" "))
print(words)

## Parse Medication Leaflets from RavimRegister.ee

In [ ]:
for subdir in ["SPC"]:#, "PIL"]:
    for pdf in tqdm(list_files_in_directory(f"../data/ravimregister/{subdir}/pdf")):
        filename = os.path.basename(pdf)
        filename, _ = os.path.splitext(filename)
        if not os.path.exists(f"../data/ravimregister/{subdir}/txt/{filename}.txt"):
            text = extract_text_from_pdf(pdf, use_gemini=False)
            with open(f"../data/ravimregister/{subdir}/txt/{filename}.txt", "w") as f:
                f.write(text)

# Prepare bilingual translations

In [ ]:
bilingual_sources = [
    "ATC.csv",
    "ContSys.csv",
    "ICD10.csv",
    "ICD11.csv",
    "LOINC.csv",
    "Nomesco.csv",
    "Substances.csv",
    "WHO.csv",
]

bilingual_df = pd.concat([
    pd.read_csv("../data/EE-EN/" + csv)[["EN", "EE"]] for csv in bilingual_sources
]).dropna().drop_duplicates()

print(bilingual_df.shape)

bilingual_df.to_csv("../data/EE-EN/all_bilingual_pairs.csv", index=False)

## EMEA corpus from OPUS

In [ ]:
def parse_tmx(tmx_file) -> Iterator[Tuple[str, str]]:
    """
    Parse a TMX file and yield tuples of (English, Estonian) text pairs.
    
    Args:
        tmx_file: File path as string or file-like object
        
    Yields:
        Tuples of (English text, Estonian text)
    """
    # Parse the XML file
    tree = ET.parse(tmx_file)
    root = tree.getroot()
    
    # Iterate through all translation units
    for tu in root.findall('.//tu'):
        en_text = None
        et_text = None
        
        # Find the English and Estonian segments
        for tuv in tu.findall('./tuv'):
            lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
            
            # Extract the text from the segment
            seg = tuv.find('./seg')
            if seg is not None and seg.text is not None:
                if lang == 'en':
                    en_text = seg.text
                elif lang == 'et':
                    et_text = seg.text
        
        # Only yield if we have both languages
        if en_text is not None and et_text is not None:
            yield (en_text, et_text)

# Chroma Indexing

Execute `chroma run` to start Chroma.

In [ ]:
def insert_document_to_chroma(document_text: str, metadata: dict, doc_id: str, collection_name: str):
    text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=200) # the defaults
    chunks = text_splitter.split_text(document_text)
    client = chromadb.PersistentClient()
    collection = client.get_or_create_collection(name=collection_name)
    
    # Create unique ids for each chunk (e.g., "doc123_0", "doc123_1", ...)
    chunk_ids = [f"{doc_id}_{i}" for i in range(len(chunks))]
    
    
    # Insert all chunks in batch, reusing the same metadata for each chunk.
    collection.add(
        documents=chunks,
        metadatas=[metadata] * len(chunks),
        ids=chunk_ids
    )
    
    print(f"Inserted {len(chunks)} chunks into the '{collection_name}' collection.")

## Build the index of the sonaveeb.ee dictionaries

In [ ]:
client = chromadb.PersistentClient()
collection = client.get_or_create_collection("sonaveeb")
sonaveeb_df = pd.read_csv("../data/sonaveeb.csv")

for row in tqdm(sonaveeb_df.dropna().itertuples(), total=sonaveeb_df.shape[0]):
    collection.upsert(documents=[row.desc], metadatas=[{"lang": row.lang}], ids=[row.term])

## Build the EMEA index

In [ ]:
chunk_size = 100
client = chromadb.PersistentClient()
collection = client.get_or_create_collection("emea")

for idx, pairs in tqdm(enumerate(chunked(parse_tmx("../data/emea/en-et.tmx"), chunk_size))):
    retries = 0
    if idx >= 455:
        while retries <= 5:
            en, et = zip(*pairs)
            try:
                collection.add(
                    documents=list(en),
                    metadatas=[{"direction": "EN->ET"} for _ in en],
                    ids=list(et)
                )
                break
            except:                
                retries += 1
                time.sleep(1)
                print(f"IDX {idx} retries {retries}")
                if retries >= 5:
                    raise Exception

## Build the index of translation pairs

In [ ]:
client = chromadb.PersistentClient()
client.delete_collection(name="paired_translations")

In [ ]:
check_key = lambda collection, key: len(collection.get(ids=[key])["ids"]) > 0

In [ ]:
bilingual_df = pd.read_csv("../data/EE-EN/all_bilingual_pairs.csv")
client = chromadb.PersistentClient()
collection = client.get_or_create_collection(name="paired_translations")

chunk_size = 100
n_iter = bilingual_df.shape[0] // chunk_size
abandonned = list()
id_iter = count(1)

for idx, rows in tqdm(enumerate(chunked(bilingual_df.itertuples(), chunk_size)), total=n_iter):
    retries = 0
    if idx > 0:
        while retries <= 3:
            try:
                # Add ET -> EN    
                et_to_en = pd.DataFrame(rows)[["EE", "EN"]].dropna().groupby("EE").first().reset_index().itertuples()
                et_to_en = [r for r in et_to_en if not check_key(collection, r.EE)]    
                if len(et_to_en) > 0:
                    collection.add(
                        documents=[r.EE for r in et_to_en],
                        metadatas=[
                            {
                                "direction": "ET->EN",
                                "translation": r.EN
                            } 
                            for r in et_to_en
                        ],
                        ids=[f"{next(id_iter)}_ET" for _ in et_to_en]
                    )
                # Add EN -> ET
                en_to_et = pd.DataFrame(rows)[["EE", "EN"]].dropna().groupby("EN").first().reset_index().itertuples()
                en_to_et = [r for r in en_to_et if not check_key(collection, r.EN)]    
                if len(en_to_et) > 0:
                  collection.add(
                        documents=[r.EN for r in en_to_et],
                        metadatas=[
                            {
                                "direction": "EN->EE",
                                "translation": r.EE
                            } 
                            for r in en_to_et
                        ],
                        ids=[f"{next(id_iter)}_EN" for _ in en_to_et]
                    )
                break
            except Exception as e:
                print(e)                
                retries += 1
                time.sleep(1)
                print(f"IDX {idx} retries {retries}")
                if retries >= 3:
                    print(f"Abandonned IDX {idx}")
                    abandonned += list(rows)
                    
print(f"Expecting {2*bilingual_df.shape[0]} documents; got {collection.count()}")
print(f"Abandonned {len(abandonned)} entries")

## Built the index of patient leaflets from haiglateliit.ee

In [ ]:
for idx, filename in tqdm(enumerate(os.listdir("../data/haiglateliit"))):
    if idx >= 0:
        if filename.endswith('.txt'):
            with open(f"../data/haiglateliit/{filename}", "r") as f:
                text = f.read()
                insert_document_to_chroma(
                    document_text=text, 
                    metadata={"filename": filename}, 
                    doc_id=filename,
                    collection_name="haiglateliit"
                )

## Build the index of articles from Eesti Arst

In [ ]:
for idx, filename in tqdm(enumerate(os.listdir("../data/eesti_arst"))):
    if idx > 5611:
        if filename.endswith('.txt'):
            with open(f"../data/eesti_arst/{filename}", "r") as f:
                text = f.read()
                insert_document_to_chroma(
                    document_text=text, 
                    metadata={"filename": filename}, 
                    doc_id=filename,
                    collection_name="eesti_arst"
                )

## Build the index of laboratory guidelines from kliinikum.ee

In [ ]:
client = chromadb.PersistentClient()
client.delete_collection(name="kliinikum")

for idx, filename in tqdm(enumerate(os.listdir("../data/kliinikum"))):
    if filename.endswith('.txt'):
        with open(f"../data/kliinikum/{filename}", "r") as f:
            text = f.read()
            insert_document_to_chroma(
                document_text=text, 
                metadata={"filename": filename}, 
                doc_id=filename,
                collection_name="kliinikum"
            )

## Build the index of RevimRegister leaflets

In [ ]:
client = chromadb.PersistentClient()
client.delete_collection(name="ravimregister")

for subdir in ["SPC"]:#, "PIL"]:
    for idx, filepath in tqdm(enumerate(list_files_in_directory(f"../data/ravimregister/{subdir}/txt", "txt"))):
        if idx >= 531:
            with open(filepath, "r") as f:
                text = f.read()
                try:
                    insert_document_to_chroma(
                        document_text=text, 
                        metadata={"filename": filepath}, 
                        doc_id=filepath,
                        collection_name="kliinikum"
                    )            
                except Exception as e:
                    print(f"Error inserting document {filepath}: {e}")

# Build Index for EE National Extension

In [ ]:
snomed_ee_df = pd.read_csv(
    "../data/SNOMED_EE_national_extension/xsct2_Description_Snapshot-et_EE1000181_20250530.txt", 
    delimiter="\t"
)
snomed_ee_df = snomed_ee_df[snomed_ee_df.active == 1]
PATH_TO_SERIALIZED_SNOMED_GRAPH = "../data/snomed_graph/full_concept_graph.gml"
SG = SnomedGraph.from_serialized(PATH_TO_SERIALIZED_SNOMED_GRAPH)
client = chromadb.PersistentClient()
collection = client.get_or_create_collection("paired_translations")

In [ ]:
en_added = 0
et_added = 0

for idx, row in tqdm(enumerate(snomed_ee_df.itertuples())):
    if idx >= 0:
        try:
            concept: SnomedConceptDetails = SG.get_concept_details(row.conceptId)
            fsn = concept.fsn.replace(f" ({concept.hierarchy})", "")
        except KeyError:
            pass
        else:
            try:
                if collection.get(fsn)["ids"] == []:
                    collection.add(
                        documents=[fsn],
                        metadatas=[{"direction": "EN->ET", "translation": row.term}],
                        ids=[f"{row.conceptId}_EN"]
                    )
                    en_added += 1
            except Exception as e:
                print(row.conceptId)
            # Prevents duplicate IDs.
            try:
                if collection.get(row.term)["ids"] == []:
                    collection.add(
                        documents=[row.term],
                        metadatas=[{"direction": "ET->EN", "translation": fsn}],
                        ids=[f"{row.conceptId}_ET"]
                    )
                    et_added += 1
            except Exception as e:
                print(row.conceptId)

print(f"EN added: {en_added}")
print(f"ET added: {et_added}")

# Check Collections

In [ ]:
client = chromadb.PersistentClient(path="/home/willh/Dropbox/Code/snomed/snomed_translation_v2/chroma/")

for c in client.list_collections():
    collection = client.get_collection(c.name)
    print(f"{collection.name}:\t {collection.count()}")
    

# Test retrieval

In [ ]:
client = chromadb.PersistentClient(path="/home/willh/Dropbox/Code/snomed/snomed_translation_v2/chroma/")
collection = client.get_or_create_collection("paired_translations")
collection.query(
    query_texts=["diabeet"],
    n_results=10,
    where={"direction": "ET->EN"},
)    
